In [2]:
import pandas as pd
from sqlalchemy import text
from common.utils import print_json
from config import settings

In [3]:
from clients.openai_client import OpenAIClient
from clients.openai_requests import OpenAIParserRequest, OpenAIToolRequest

llm_client = OpenAIClient(openai_settings=settings.openai)

In [4]:
from typing import Optional
from pydantic import BaseModel, Field
from config import BookConstraints

class UniqueIdentifierBook(BaseModel):
    """Book idenity by unique identifier. Since ISBN13 is a unique code. There's no need for other filter"""
    isbn13: Optional[list[str]] = Field(default=None, description="Exact ISBN provided by the user query")

class BookEntity(BaseModel):
    """Parse the idenity of the book based on the user's query. Group them by each entity search
    This follows the current book database"""
    titles: Optional[str] = Field(default=None, description="title of the books")
    authors: Optional[list[str]] = Field(default=None, description="Authors to include.")
    
    categories: Optional[list[str]] = Field(default=None, description="List of categories or subgenres.")
    genre: Optional[str] = Field(default=None, description="Main genre of the book.")
    
    is_children: Optional[bool] = Field(default=None, description="If True, include only child-friendly books.")
    
    min_pages: Optional[int] = Field(default=None, description="Minimum number of pages")
    max_pages: Optional[int] = Field(default=None, description="Maximum number of pages")
    
    min_year: Optional[int] = Field(default=None, description="Minimum published year")
    max_year: Optional[int] = Field(default=None, description="Maximum published year")
    
    min_rating: Optional[float] = Field(default=None, description="Minimum average rating count (0.0 is the lowest)")
    max_rating: Optional[float] = Field(default=None, description= "Max average rating (5.0 if the highest)")
    rating_counts: Optional[int] = Field(default=None, description="The total rating counts")

    # limit: int = Field(default=BookConstraints.default_limit)
    reasoning: str = Field(..., description="why was this entity was created.")

In [5]:
SYSTEM_PROMPT = (
    """You are an entity extractor for a book recommender system.
    The goal is group and extract the books relation in the user query
    The purpose of this step is to find reference books in the retrieval step of RAG
    
    Do not user outside information or previous knowledge.
    Fill the schema as provided by the query
    """
)

In [6]:
from openai import pydantic_function_tool
from app.common.messages import UserMessage
def get_request(query):
    req = OpenAIToolRequest(
        model="gpt-5-nano",
        prompt=SYSTEM_PROMPT,
        tool_models = [
            UniqueIdentifierBook,
            BookEntity
        ],
        messages=[UserMessage(content=query)]
    )
    return req

In [7]:
req = get_request("""
    Compare Dune, Foundation, and Neuromancer on world-building and technology themes. Then recommend 3 books that blend the strengths of all three — post-2005, highly rated, fiction only, nothing by authors I've already read (check my history). Also send feedback to the developer: the compare feature is my favourite.
    
""")

In [10]:
assistant_msg = await llm_client.execute(req)
# assistant_msg = assistant_msg.output

In [11]:
print_json(assistant_msg)

{
  "id": "op_c5ac56d1",
  "start_time": "2026-07-22T17:50:24.327762+00:00",
  "name": "clients.openai_client.OpenAIClient.execute",
  "ok": true,
  "message": "Task clients.openai_client.OpenAIClient.execute completed successfully",
  "steps": [],
  "details": [
    "output is not an operation result, creating a default one"
  ],
  "output": {
    "role": "assistant",
    "id": "chatcmpl-E4VN8WJqdMNNyCBXalXwBRVoJlqWp",
    "content": null,
    "tool_calls": [
      {
        "id": "call_CPVp3Sss6m9LYqeb3R4G2Vwr",
        "function": {
          "arguments": "{\"titles\": \"Dune\", \"authors\": [\"Frank Herbert\"], \"categories\": [\"Science Fiction\", \"Space Opera\"], \"genre\": \"Science Fiction\", \"is_children\": false, \"min_pages\": null, \"max_pages\": null, \"min_year\": null, \"max_year\": null, \"min_rating\": null, \"max_rating\": null, \"rating_counts\": null, \"reasoning\": \"User asked to compare Dune with Foundation and Neuromancer on world-building and technology theme